# 베어링 진동 FFT 분석 — CWRU 데이터셋

**목표:** 정상 베어링 vs. 결함 베어링의 진동 신호를 FFT로 분석해서 결함 주파수 찾기

**데이터:** Case Western Reserve University (CWRU) Bearing Dataset
- 12kHz 샘플링, 드라이브엔드 베어링
- 정상(Normal) / 외륜결함(Outer Race Fault) 비교

---
**베어링 결함 주파수 공식 (외륜 BPFO):**

> BPFO = (n/2) × RPM/60 × (1 - Bd/Pd × cos θ)

CWRU 6205-2RS 베어링 기준:
- 볼 수(n) = 9, 볼지름(Bd) = 0.3126", 피치지름(Pd) = 1.537", 접촉각(θ) = 0°
- 1797 RPM → **BPFO ≈ 107.4 Hz**

In [ ]:
# ── 셀 1-b: 한글 폰트 설치 (Colab 전용) ──────────────────────────────────────
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)

import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지
print('한글 폰트 설정 완료 — 이후 셀부터 적용됨')

In [ ]:
# ── 셀 1: 라이브러리 설치 및 임포트 ──────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import scipy.io
import requests

print('라이브러리 로드 완료')

In [ ]:
# ── 셀 2: CWRU 데이터 다운로드 ────────────────────────────────────────────────
# 정상 베어링 (Normal, 0HP 부하)
url_normal = 'https://engineering.case.edu/sites/default/files/97.mat'
# 외륜결함 베어링 (Outer Race Fault, 0.007inch, 0HP)
url_fault  = 'https://engineering.case.edu/sites/default/files/105.mat'

def download_mat(url, filename):
    r = requests.get(url, timeout=30)
    with open(filename, 'wb') as f:
        f.write(r.content)
    print(f'다운로드 완료: {filename} ({len(r.content)//1024} KB)')

download_mat(url_normal, 'normal.mat')
download_mat(url_fault,  'fault_or.mat')

In [ ]:
# ── 셀 3: 데이터 로드 & 구조 확인 ────────────────────────────────────────────
normal_mat = scipy.io.loadmat('normal.mat')
fault_mat  = scipy.io.loadmat('fault_or.mat')

print('=== 정상 데이터 키 목록 ===')
print([k for k in normal_mat.keys() if not k.startswith('_')])

print('\n=== 결함 데이터 키 목록 ===')
print([k for k in fault_mat.keys() if not k.startswith('_')])

In [ ]:
# ── 셀 4: 드라이브엔드 가속도 신호 추출 ──────────────────────────────────────
# CWRU 키 이름: X097_DE_time(정상), X105_DE_time(결함) — 실제 키에 맞게 조정
# 아래는 DE(Drive End) 채널을 자동으로 찾는 코드

def get_de_signal(mat_dict):
    for key in mat_dict.keys():
        if 'DE_time' in key:
            return mat_dict[key].flatten()
    raise KeyError('DE_time 키를 찾지 못했습니다. 키 목록을 확인하세요.')

sig_normal = get_de_signal(normal_mat)
sig_fault  = get_de_signal(fault_mat)

Fs = 12000  # 샘플링 주파수 (Hz)
print(f'정상 신호 길이: {len(sig_normal):,} 샘플 ({len(sig_normal)/Fs:.1f}초)')
print(f'결함 신호 길이: {len(sig_fault):,} 샘플 ({len(sig_fault)/Fs:.1f}초)')

In [ ]:
# ── 셀 5: 시간 영역 비교 시각화 ───────────────────────────────────────────────
N_plot = int(0.1 * Fs)  # 0.1초만 보기
t = np.arange(N_plot) / Fs

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(t, sig_normal[:N_plot], color='steelblue', linewidth=0.8)
axes[0].set_title('정상 베어링 — 시간 영역 (0.1초)', fontsize=13)
axes[0].set_ylabel('가속도 (g)')
axes[0].set_xlabel('시간 (s)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, sig_fault[:N_plot], color='tomato', linewidth=0.8)
axes[1].set_title('외륜결함 베어링 — 시간 영역 (0.1초)', fontsize=13)
axes[1].set_ylabel('가속도 (g)')
axes[1].set_xlabel('시간 (s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('time_domain.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ 시간 영역만 봐서는 어디가 다른지 잘 모름. FFT 해야 명확해짐.')

In [ ]:
# ── 셀 6: FFT 함수 정의 ───────────────────────────────────────────────────────
def compute_fft(signal, fs, freq_max=1000):
    """
    signal : 1D 시계열
    fs     : 샘플링 주파수 (Hz)
    freq_max: 표시할 최대 주파수 (Hz)
    반환: (freqs, magnitude) — 단측 스펙트럼
    """
    N = len(signal)
    fft_vals = np.fft.rfft(signal)
    freqs    = np.fft.rfftfreq(N, d=1/fs)
    magnitude = (2.0 / N) * np.abs(fft_vals)  # 단측 진폭 스펙트럼
    mask = freqs <= freq_max
    return freqs[mask], magnitude[mask]

print('FFT 함수 정의 완료')

In [ ]:
# ── 셀 7: FFT 계산 & 주파수 영역 비교 ────────────────────────────────────────
# 결함 주파수 (CWRU 6205-2RS, 1797RPM)
BPFO = 107.4  # 외륜결함 주파수 (Hz)

f_norm,  mag_norm  = compute_fft(sig_normal, Fs, freq_max=600)
f_fault, mag_fault = compute_fft(sig_fault,  Fs, freq_max=600)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# 정상
axes[0].plot(f_norm, mag_norm, color='steelblue', linewidth=0.8)
axes[0].set_title('정상 베어링 — FFT 주파수 영역', fontsize=13)
axes[0].set_ylabel('진폭 (g)')
axes[0].set_xlabel('주파수 (Hz)')
axes[0].grid(True, alpha=0.3)

# 결함 + BPFO 표시
axes[1].plot(f_fault, mag_fault, color='tomato', linewidth=0.8)
for harmonic in [1, 2, 3]:  # BPFO 고조파
    axes[1].axvline(BPFO * harmonic, color='gold', linestyle='--', linewidth=1.5,
                    label=f'{harmonic}×BPFO = {BPFO*harmonic:.1f} Hz' if harmonic == 1 else f'{harmonic}×BPFO')
axes[1].set_title('외륜결함 베어링 — FFT (노란 점선: BPFO 고조파)', fontsize=13)
axes[1].set_ylabel('진폭 (g)')
axes[1].set_xlabel('주파수 (Hz)')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fft_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ BPFO 위치에 봉우리가 있으면 외륜결함 확인!')

In [ ]:
# ── 셀 8: 결함 주파수 부근 피크 탐색 ─────────────────────────────────────────
def find_peak_near(freqs, magnitude, target_hz, window_hz=5):
    """target_hz ± window_hz 범위에서 최대 진폭과 주파수 반환"""
    mask = (freqs >= target_hz - window_hz) & (freqs <= target_hz + window_hz)
    if mask.sum() == 0:
        return None, None
    idx = np.argmax(magnitude[mask])
    return freqs[mask][idx], magnitude[mask][idx]

print('=== 결함 신호에서 BPFO 고조파 피크 탐색 ===')
for h in [1, 2, 3]:
    freq_peak, mag_peak = find_peak_near(f_fault, mag_fault, BPFO * h)
    if freq_peak is not None:
        print(f'  {h}×BPFO ({BPFO*h:.1f} Hz) → 실측 피크: {freq_peak:.1f} Hz, 진폭: {mag_peak:.4f} g')

print('\n=== 정상 신호에서 동일 위치 비교 ===')
for h in [1, 2, 3]:
    freq_peak, mag_peak = find_peak_near(f_norm, mag_norm, BPFO * h)
    if freq_peak is not None:
        print(f'  {h}×BPFO ({BPFO*h:.1f} Hz) → 진폭: {mag_peak:.4f} g')

---
## Part 2. 엔벨로프 분석 (Envelope Analysis)

**왜 필요한가?**  
raw FFT에서는 축 회전 고조파·전기 노이즈가 BPFO 근처에 섞여 판별이 불명확했다.  
엔벨로프 분석은 **충격 성분만 뽑아내서** FFT를 다시 하는 방법이다.

**3단계 프로세스:**
1. **대역통과 필터(Bandpass)** — 베어링 공진 대역(2000~5000Hz)만 남김
2. **포락선 추출(Hilbert transform)** — 필터링된 신호에서 충격의 크기 변화만 추출
3. **엔벨로프 FFT** — 추출된 포락선에 FFT 적용 → BPFO 봉우리 선명하게 나타남

In [ ]:
# ── 셀 9: 엔벨로프 분석 함수 정의 ────────────────────────────────────────
from scipy.signal import butter, filtfilt, hilbert

def envelope_analysis(signal, fs, f_low=2000, f_high=5000, freq_max=600):
    """
    signal : 1D 시계열
    fs     : 샘플링 주파수
    f_low ~ f_high : 베어링 공진 대역 (Hz)
    반환   : (freqs, envelope_spectrum)
    """
    # 1단계: 대역통과 필터 — 공진 대역만 남기기
    nyq = fs / 2
    b, a = butter(4, [f_low/nyq, f_high/nyq], btype='band')
    filtered = filtfilt(b, a, signal)

    # 2단계: Hilbert transform → 포락선(충격 크기 변화) 추출
    envelope = np.abs(hilbert(filtered))

    # 3단계: 포락선에 FFT
    envelope_zero = envelope - np.mean(envelope)  # DC 제거
    freqs, magnitude = compute_fft(envelope_zero, fs, freq_max=freq_max)
    return freqs, magnitude

print('엔벨로프 분석 함수 정의 완료')

In [ ]:
# ── 셀 10: 엔벨로프 스펙트럼 계산 & 시각화 ───────────────────────────────
f_env_norm,  mag_env_norm  = envelope_analysis(sig_normal, Fs)
f_env_fault, mag_env_fault = envelope_analysis(sig_fault,  Fs)

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(f_env_norm, mag_env_norm, color='steelblue', linewidth=0.8)
axes[0].set_title('정상 베어링 — 엔벨로프 스펙트럼', fontsize=13)
axes[0].set_ylabel('진폭 (g)')
axes[0].set_xlabel('주파수 (Hz)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(f_env_fault, mag_env_fault, color='tomato', linewidth=0.8)
for h in [1, 2, 3, 4]:
    lbl = f'{h}xBPFO = {BPFO*h:.1f} Hz' if h == 1 else f'{h}xBPFO'
    axes[1].axvline(BPFO * h, color='gold', linestyle='--', linewidth=1.5, label=lbl)
axes[1].set_title('외륜결함 베어링 — 엔벨로프 스펙트럼 (노란 점선: BPFO 고조파)', fontsize=13)
axes[1].set_ylabel('진폭 (g)')
axes[1].set_xlabel('주파수 (Hz)')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('envelope_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ raw FFT보다 BPFO 봉우리가 훨씬 선명하게 나타남')

In [ ]:
# ── 셀 11: raw FFT vs 엔벨로프 — BPFO 진폭 최종 비교 ────────────────────
print('=== BPFO 1x 위치 진폭 비교 ===')
print(f'{"방법":20s} {"정상":>12s} {"결함":>12s} {"결함/정상 배율":>14s}')
print('-' * 62)

for label, f_n, m_n, f_f, m_f in [
    ('raw FFT',   f_norm,     mag_norm,     f_fault,     mag_fault),
    ('엔벨로프', f_env_norm, mag_env_norm, f_env_fault, mag_env_fault),
]:
    _, amp_n = find_peak_near(f_n, m_n, BPFO, window_hz=5)
    _, amp_f = find_peak_near(f_f, m_f, BPFO, window_hz=5)
    ratio = amp_f / amp_n if (amp_n and amp_n > 0) else float('nan')
    print(f'{label:20s} {amp_n:>12.5f} {amp_f:>12.5f} {ratio:>14.1f}x')

print('\n→ 엔벨로프에서 결함/정상 배율이 클수록 진단 신뢰도 높음')

## 최종 결론

| 방법 | BPFO 판별 | 특징 |
|---|---|---|
| raw FFT | 불명확 (노이즈 오염) | 빠르지만 베어링 초기결함 부적합 |
| **엔벨로프 분석** | **선명** | ISO 18436-2 현장 표준, 초기결함 검출 강점 |

**포트폴리오 스토리:**
> raw FFT 시도 → 한계 발견 → 엔벨로프 분석으로 개선 → BPFO 선명 추출 성공

이 흐름이 **"코드만 돌린 게 아니라 왜 안 되는지 이해하고 개선한 사람"** 을 증명한다.

## 결과 해석

- **정상 신호**: BPFO 부근에 뚜렷한 봉우리 없음
- **결함 신호**: BPFO (≈107 Hz) 및 고조파(2×, 3×)에서 봉우리 확인
- **결론**: FFT 한 장으로 "외륜에 흠집 있음" 진단 가능

---
**다음 단계:**
1. 내륜결함(BPFI), 볼결함(BSF)도 같은 방식으로 비교
2. RMS / Kurtosis 등 시간 영역 특징량 추가
3. ML(분류기)로 4가지 상태 자동 분류 → 포트폴리오 1호 완성